# SpatioTemporal

- Cacular a zona de pixels baseado no espectro de ndvi e evi, utilizando reducer 

In [1]:
import ee

from repository_ee import (
    PROJECT,
    Exports,
    Modis,
    PeriodoAgricola,
    initialize_earth_engine,
)

initialize_earth_engine(project=PROJECT)

In [2]:
ASSET_SPATIO_STUDY = (
    "projects/lateral-spirit-514212-j0/assets/Amostras_PASTAGEM_WIDE_2025"
)
ANO = 2025
regioes = ee.FeatureCollection(ASSET_SPATIO_STUDY).limit(1)
area_estudo = regioes.first().geometry()

In [3]:
periodo = PeriodoAgricola.tocantins(ANO)
modis = Modis.collection(area_estudo, period=periodo).sort("system:time_start")

In [4]:
data_inicio = ee.Date(periodo.inicio_anual)
intervalo = 1
unidade_intervalo = "month" #Configuração temporal mensal
# A colecao MODIS do ano selecionado e dividida em 12 janelas mensais.
quantidade_intervalos = 12
redutor_temporal = ee.Reducer.mean()

# Redutor

- Reduz para apenas uma imagem uma coleção de imagens de uma área de estudo que foram capturadas em um determinado tempo.
- Empilha valores como .median() .mean() .max(). E transformar todos esses valores em apenas 1 valor final para aquele pixel
- assim acumulando e tornando a coleção de imagens e uma única imagem representativa
- facilitando a organização dos dados em um só lugar para fazer calculos
- Facilita a mascaração dos pixel , reduzindo ruídos como nuvems
- Facilitar as metricas fenológicas.

In [5]:
redutor_espacial = (
    ee.Reducer.mean()
    .combine(reducer2=ee.Reducer.stdDev(), sharedInputs=True)
    .combine(reducer2=ee.Reducer.variance(), sharedInputs=True)
)

# LONG TABLE
- Na prática, o formato Long Table (tabela longa ou empilhada) organiza os seus dados de modo que cada linha represente uma única observação no tempo e no espaço — neste caso, a combinação de 1 Região × 1 Janela de Tempo.

- Em vez de criar dezenas de colunas para as datas (ex: NDVI_Jan, NDVI_Fev, NDVI_Mar — o que seria uma Wide Table), você mantém poucas colunas fixas e "empilha" os resultados um embaixo do outro conforme o tempo passa.


In [6]:
"""Monta a tabela longa de estatísticas zonais de NDVI no Earth Engine.

A tabela tem uma linha por região e intervalo, com a data de início do
composto e a média, o desvio padrão e a variância dos pixels da região.
As tarefas preparam um CSV no Google Drive e um asset de tabela no
Earth Engine. Chame start() na tarefa do destino desejado.
"""

indices_intervalos = ee.List.sequence(0, quantidade_intervalos - 1)
projecao = modis.first().projection()
escala = projecao.nominalScale()

# A função calcular_estatisticas_zonais roda uma vez para cada número da lista indices_intervalos.
# Em cada volta, ela avança a data (inicio e fim), filtra o MODIS e aplica o redutor_temporal, gerando 1 imagem composta limpa para aquele intervalo.
def calcular_estatisticas_zonais(indice: ee.Number) -> ee.FeatureCollection:
    """Calcula as estatísticas por região para a janela indicada pelo índice.

    O índice começa em zero e determina o deslocamento em relação a
    data_inicio. A função calcula a média temporal por pixel e resume
    o composto em cada região na projeção e na escala nativas do MODIS.
    """
    inicio = data_inicio.advance(
        ee.Number(indice).multiply(intervalo), unidade_intervalo
    )
    fim = inicio.advance(intervalo, unidade_intervalo)
    composto = modis.filterDate(inicio, fim).reduce(redutor_temporal)

    estatisticas = composto.reduceRegions(
        collection=regioes,
        reducer=redutor_espacial,
        scale=escala,
        crs=projecao,
    )

    def adicionar_data(feicao: ee.Feature) -> ee.Feature:
        """Registra o início da janela em composite_start no formato AAAA-MM-DD."""
        return feicao.set("composite_start", inicio.format("yyyy-MM-dd"))

    return estatisticas.map(adicionar_data)


estatisticas_zonais_long = ee.FeatureCollection(
    indices_intervalos.map(calcular_estatisticas_zonais)
).flatten()

nome_exportacao_long = f"zonal_stats_long_PASTAGEM{ANO}"
tarefa_drive_long = Exports.table_to_drive(
    collection=estatisticas_zonais_long,
    description=nome_exportacao_long,
)
tarefa_asset_long = Exports.table_to_assets(
    collection=estatisticas_zonais_long,
    asset_id=f"projects/{PROJECT}/assets/{nome_exportacao_long}",
    description=nome_exportacao_long,
)
#tarefa_drive_long.start()
tarefa_asset_long.start()
estatisticas_zonais_long

# Wide table

In [ ]:
"""Monta uma tabela com uma linha por região e colunas por estatística e data.

Cada janela usa a média temporal do NDVI por pixel e os mesmos redutores
espaciais da Long Table. A data de início distingue as colunas de cada janela.
As tarefas preparam um CSV no Google Drive e um asset de tabela no
Earth Engine. Chame start() na tarefa do destino desejado.
"""


def adicionar_estatisticas_da_janela(
    indice: ee.Number, regioes_acumuladas: ee.FeatureCollection
) -> ee.FeatureCollection:
    """Acrescenta as estatísticas da janela às propriedades de cada região."""
    inicio = data_inicio.advance(
        ee.Number(indice).multiply(intervalo), unidade_intervalo
    )
    fim = inicio.advance(intervalo, unidade_intervalo)
    composto = modis.filterDate(inicio, fim).reduce(redutor_temporal)
    data_composto = inicio.format("yyyy-MM-dd")

    def adicionar_colunas(feicao: ee.Feature) -> ee.Feature:
        """Resume o composto e acrescenta a data aos nomes das estatísticas."""
        estatisticas = composto.reduceRegion(
            reducer=redutor_espacial,
            geometry=feicao.geometry(),
            scale=escala,
            crs=projecao,
        )
        nomes = estatisticas.keys()
        nomes_datados = nomes.map(
            lambda nome: ee.String(nome).cat("_").cat(data_composto)
        )
        return feicao.set(estatisticas.rename(nomes, nomes_datados))

    return ee.FeatureCollection(regioes_acumuladas).map(adicionar_colunas)


estatisticas_zonais_wide = ee.FeatureCollection(
    indices_intervalos.iterate(adicionar_estatisticas_da_janela, regioes)
)

nome_exportacao_wide = f"zonal_stats_wide_PASTAGEM{ANO}"
tarefa_drive_wide = Exports.table_to_drive(
    collection=estatisticas_zonais_wide,
    description=nome_exportacao_wide,
)
tarefa_asset_wide = Exports.table_to_assets(
    collection=estatisticas_zonais_wide,
    asset_id=f"projects/{PROJECT}/assets/{nome_exportacao_wide}",
    description=nome_exportacao_wide,
)
#tarefa_drive_wide.start()
#tarefa_asset_wide.start()
estatisticas_zonais_wide